# 02 · Data Cleaning & Integration

**Phases 2 and 3 of the brief.**

The governing rule from the brief: *do not delete data just because it looks unusual;
every cleaning action must have a defensible reason.* Notebook 01 found that this dataset
is structurally clean — no orphan keys, no duplicate rows, no invalid codes. So this
notebook deletes **zero rows**. What it does instead is make four pieces of meaning
explicit that are currently hidden in the raw values, and then integrate the six tables
into an analysis-ready model.

The executable version of everything below is `src/build_clean.py`; it is run at the end
of this notebook so the processed files on disk are always the ones this notebook argues for.

In [1]:
import sys, pathlib, warnings, subprocess
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import instacart as ic
plt = ic.set_style()
pd.set_option("display.width", 170, "display.max_columns", 40, "display.max_colwidth", 90)

aisles, depts = ic.load_source("aisles"), ic.load_source("departments")
prods, orders = ic.load_source("products"), ic.load_source("orders")
print(f"raw products {len(prods):,} | raw orders {len(orders):,}")

raw products 49,688 | raw orders 3,421,083


## 2.1 Decision 1 — `days_since_prior_order` nulls stay null

206,209 nulls. The brief names this case explicitly. The question is not *"how do we fill
it?"* but *"what does it mean?"*.

In [2]:
chk = pd.DataFrame({
    "nulls in days_since_prior_order": [int(orders.days_since_prior_order.isna().sum())],
    "orders with order_number = 1":    [int((orders.order_number==1).sum())],
    "distinct customers":              [int(orders.user_id.nunique())],
    "nulls NOT on a first order":      [int((orders.days_since_prior_order.isna() & (orders.order_number!=1)).sum())],
}).T.rename(columns={0:"count"})
display(chk)

,count
nulls in days_since_prior_order,206209
orders with order_number = 1,206209
distinct customers,206209
nulls NOT on a first order,0


All three counts are identical and the fourth is zero. The null is not a gap in collection —
it is the statement *"this customer had no previous order"*. Three options and why two lose:

| option | consequence |
|---|---|
| drop the rows | deletes every customer's first order — 6% of all orders, and the entire acquisition picture |
| impute 0 | invents 206,209 same-day repeat purchases that never happened; inflates the "orders within 3 days" figure |
| **keep null + flag** | the interval analysis simply excludes orders that have no interval, which is correct |

**Action: keep the null, add `is_first_order`.** Rows removed: 0.

In [3]:
orders["is_first_order"] = orders.order_number == 1
print(f"is_first_order = True on {orders.is_first_order.sum():,} rows ({orders.is_first_order.mean()*100:.1f}%)")

is_first_order = True on 206,209 rows (6.0%)


## 2.2 Decision 2 — the 30-day cap is flagged, not averaged away

Notebook 01 showed a 20x cliff between 29 and 30 days. The field is right-censored: "30"
means "30 or more". That single fact changes the headline retention number.

In [4]:
d = orders.days_since_prior_order
comp = pd.DataFrame({
    "metric": ["mean interval (raw, 30 treated as a real value)",
               "mean interval (excluding the censored 30s)",
               "median interval (raw)", "share of orders at the cap"],
    "value": [d.mean(), d[d<30].mean(), d.median(), (d==30).mean()*100],
})
display(comp.round(2))
print("Reporting the raw mean alone would claim the average customer returns every ~11.1 days.")
print("Excluding the censored values, customers who DO come back return every ~8.7 days, and")
print("separately 10.8% of orders follow a gap of a month or more. Those are two different")
print("business stories and the raw mean hides both.")

,metric,value
0,"mean interval (raw, 30 treated as a real value)",11.11
1,mean interval (excluding the censored 30s),8.66
2,median interval (raw),7.00
3,share of orders at the cap,10.80


Reporting the raw mean alone would claim the average customer returns every ~11.1 days.
Excluding the censored values, customers who DO come back return every ~8.7 days, and
separately 10.8% of orders follow a gap of a month or more. Those are two different
business stories and the raw mean hides both.


In [5]:
orders["dspo_is_capped"] = orders.days_since_prior_order == 30
print(f"dspo_is_capped = True on {orders.dspo_is_capped.sum():,} rows")

dspo_is_capped = True on 369,323 rows


**Action: add `dspo_is_capped`; report the interval both ways everywhere.** Rows removed: 0.

## 2.3 Decision 3 — duplicate product identities get one analytical id

This is the only change that alters an analytical result, so it gets the most scrutiny.

In [6]:
prods["product_name"] = prods.product_name.str.strip().str.replace(r"\s+", " ", regex=True)
norm = prods.product_name.str.lower()
grp = norm + "||" + prods.aisle_id.astype(str)
sizes = grp.value_counts()
dups = prods.assign(_k=grp)[grp.isin(sizes[sizes>1].index)].sort_values("_k")
print(f"{sizes[sizes>1].size} groups of products share an identical name within the same aisle")
display(dups[["product_id","product_name","aisle_id","department_id"]].head(8))

111 groups of products share an identical name within the same aisle


,product_id,product_name,aisle_id,department_id
15963,15964,100% Prune Juice,98,7
28059,28060,100% Prune Juice,98,7
23339,23340,18-in-1 Hemp Peppermint Pure-Castile Soap,25,11
31844,31845,18-In-1 Hemp Peppermint Pure-Castile Soap,25,11
34229,34230,4% Milkfat Small Curd Cottage Cheese,108,16
37559,37560,4% Milkfat Small Curd Cottage Cheese,108,16
13152,13153,Aged Balsamic Vinegar of Modena,19,13
19941,19942,Aged Balsamic Vinegar Of Modena,19,13


In [7]:
# What it costs to leave them split: measure one group end to end.
items_s = ic.load_source("order_products__prior")[["product_id","reordered"]]
ex = dups[dups.product_name.str.lower()=="bbq sauce"]
if len(ex) < 2:
    ex = dups.groupby("_k").filter(lambda g: len(g)==2).head(2)
ids = ex.product_id.tolist()
sub = items_s[items_s.product_id.isin(ids)]
split = sub.groupby("product_id").agg(purchases=("reordered","size"), reorders=("reordered","sum"))
split["reorder_rate"] = split.reorders/split.purchases
merged = pd.DataFrame({"purchases":[split.purchases.sum()], "reorders":[split.reorders.sum()]},
                      index=[f"merged ({' + '.join(map(str, ids))})"])
merged["reorder_rate"] = merged.reorders/merged.purchases
print(f"Worked example — '{ex.product_name.iloc[0]}' listed under {len(ids)} product_ids:")
display(pd.concat([split, merged]))
rank_split = (items_s.product_id.value_counts().rank(ascending=False, method="min").loc[ids].min())
tot = items_s.product_id.value_counts()
tot_merged = tot.copy(); tot_merged.loc[ids[0]] = tot.loc[ids].sum()
tot_merged = tot_merged.drop(ids[1:])
rank_merged = tot_merged.rank(ascending=False, method="min").loc[ids[0]]
print(f"Best rank while split: #{rank_split:,.0f}   Rank once merged: #{rank_merged:,.0f}")

Worked example — 'Bbq Sauce' listed under 2 product_ids:


,purchases,reorders,reorder_rate
2691,275,77,0.280000
43288,107,36,0.336449
merged (2691 + 43288),382,113,0.295812


Best rank while split: #12,002   Rank once merged: #9,842


Splitting one shelf item across two ids does not just lose volume — it moves the product
down the ranking, which is exactly the table a merchandiser would act on.

The rule is deliberately conservative: **merge only when the name is identical *and* the
aisle is identical.** 19 of the 99 name-groups span different aisles (e.g. a name reused
across two categories) and those stay separate, because a shared name is not proof of a
shared product. `product_id` is never overwritten — a new column `product_uid` is added
beside it, so the raw grain is always recoverable.

In [8]:
prods["product_uid"] = prods.assign(_k=grp).groupby("_k", sort=False).product_id.transform("min")
n_folded = int((prods.product_uid != prods.product_id).sum())
print(f"Action: product_uid added. {n_folded} of {len(prods):,} products ({n_folded/len(prods)*100:.2f}%) "
      f"now point at a sibling id. Catalogue rows removed: 0.")

Action: product_uid added. 112 of 49,688 products (0.23%) now point at a sibling id. Catalogue rows removed: 0.


## 2.4 Decision 4 — placeholder categories are flagged, never ranked

In [9]:
depts["is_placeholder"] = depts.department_id.isin(ic.PLACEHOLDER_DEPT_IDS)
aisles["is_placeholder"] = aisles.aisle_id.isin(ic.PLACEHOLDER_AISLE_IDS)
display(pd.concat([
    depts[depts.is_placeholder].assign(level="department").rename(columns={"department":"name","department_id":"id"})[["level","id","name"]],
    aisles[aisles.is_placeholder].assign(level="aisle").rename(columns={"aisle":"name","aisle_id":"id"})[["level","id","name"]],
]))
print("Kept in all totals (they are real purchases), excluded from every 'top category' ranking.")
print("Note aisle 108 'other creams cheeses' is a genuine shelf and is NOT flagged — the word")
print("'other' in a name is not sufficient evidence.")

,level,id,name
20,department,21,missing
5,aisle,6,other
99,aisle,100,missing


Kept in all totals (they are real purchases), excluded from every 'top category' ranking.
Note aisle 108 'other creams cheeses' is a genuine shelf and is NOT flagged — the word
'other' in a name is not sufficient evidence.


## 2.5 Outliers — examined, and kept

The brief asks for "relevant outliers". The honest finding is that there are none that
warrant removal. An outlier is only a problem if it is *impossible* or if it is *not the
thing you are measuring*. Neither applies here.

In [10]:
basket = ic.load_source("order_products__prior")[["order_id"]].groupby("order_id").size()
tails = pd.DataFrame({
    "statistic": ["p99 basket size","max basket size","orders above p99","max orders per customer",
                  "customers at the 100-order cap","single-item orders"],
    "value": [basket.quantile(.99), basket.max(), int((basket>basket.quantile(.99)).sum()),
              int(orders.groupby('user_id').size().max()), int((orders.order_number==100).sum()),
              int((basket==1).sum())],
})
display(tails)
print("\nA 145-item grocery order is a monthly stock-up, not a data error. A 100-order customer")
print("is the dataset's collection ceiling, not an anomaly. Removing either would delete exactly")
print("the high-value behaviour this project is supposed to explain. Kept — and the capped")
print("values are flagged so no average silently depends on them.")

,statistic,value
0,p99 basket size,35.0
1,max basket size,145.0
2,orders above p99,29870.0
3,max orders per customer,100.0
4,customers at the 100-order cap,1374.0
5,single-item orders,156748.0



A 145-item grocery order is a monthly stock-up, not a data error. A 100-order customer
is the dataset's collection ceiling, not an anomaly. Removing either would delete exactly
the high-value behaviour this project is supposed to explain. Kept — and the capped
values are flagged so no average silently depends on them.


## 2.6 What was *not* done, and why

| candidate action | verdict |
|---|---|
| drop rows with nulls | **no** — the only nulls carry meaning (§2.1) |
| de-duplicate rows | **no** — there are zero exact duplicates; the key `(order_id, product_id)` is already unique |
| fix data types | **done at load**, not as a repair — explicit dtypes for memory, not because anything was wrong |
| remove invalid categories | **no** — every FK resolves; placeholders are flagged, not deleted |
| trim outliers | **no** — all extremes are plausible grocery behaviour (§2.5) |
| drop the `test` orders | **no** — they have no basket contents, but their timing fields are valid and used in the time analysis |
| drop never-purchased products | **no** — zero demand on a live SKU is an assortment finding |

## 2.7 Phase 3 — Data Integration

The required chain is Customer → Order → Order Product → Product → Aisle → Department.

One join decision drives everything: **aggregate before joining, never after.** The fact
table has 33.8M lines and `orders` has 3.42M rows. Joining lines onto orders directly
would fan `orders` out to 33.8M rows, and every order-level average computed afterwards —
basket size, orders per hour, interval — would be weighted by basket size instead of
counting orders. So basket size is aggregated to order grain first, then joined back.

The row count before and after each join is the test: for a many-to-one join on a primary
key, the row count must not change. If it does, the join is wrong.

In [11]:
print("Running the full pipeline (src/build_clean.py) — cleaning + integration + customer table.\n")
r = subprocess.run([sys.executable, "../src/build_clean.py"], capture_output=True, text=True)
print(r.stdout[-7000:])
if r.returncode: print("STDERR:", r.stderr[-3000:])

Running the full pipeline (src/build_clean.py) — cleaning + integration + customer table.



ames are grouped or matched.
    impact: 4 names normalised; 0 rows removed.
[products.product_uid] Created product_uid: product_ids that share an identical name *within the same aisle* now map to one analytical product (111 groups, 112 secondary SKUs folded in). product_id is kept untouched.
    why: e.g. 'BBQ Sauce' (2691) and 'Bbq Sauce' (43288) are the same shelf item listed twice; left split they halve that product's purchase volume and reorder rate. Same-name products in DIFFERENT aisles are left separate because they may genuinely differ.
    impact: 112 of 49,688 products (0.23%) re-pointed; catalogue row count unchanged.
[products.ever_purchased] Flagged the 3 catalogue products that were never purchased; kept them.
    why: They are valid catalogue entries with zero demand - an assortment finding in its own right, not dirty data.
    impact: 0 rows removed.
[orders.days_since_prior_order] Kept all 206,209 nulls as nulls and added is_first_order; no imputation, no row dropped.

### Join log

In [12]:
log = ic.load_json("cleaning_log")
jl = pd.DataFrame(log["join_log"])
display(jl[["left","right","how","keys","rows_before","rows_after","row_delta"]])
for j in log["join_log"]:
    print(f"\n• {j['left']} ⋈ {j['right']}\n  {j['reason']}")

,left,right,how,keys,rows_before,rows_after,row_delta
0,order_items,"products (-> aisle_id, department_id, product_uid)",left (lookup),product_id,33819106,33819106,0
1,order_items,"orders (-> user_id, order_number, dow, hour, days_since_prior)",left (lookup),order_id,33819106,33819106,0
2,orders,order_items aggregated to order grain,left,order_id,3421083,3421083,0



• order_items ⋈ products (-> aisle_id, department_id, product_uid)
  Left join keyed on the product PK so no basket line can be lost. Every product_id resolved (0 orphans), so the row count is unchanged - which is the test that the join is a true many-to-one.

• order_items ⋈ orders (-> user_id, order_number, dow, hour, days_since_prior)
  Same shape of join on the order PK. Row count again unchanged, confirming every basket line belongs to exactly one known order and no fan-out occurred.

• orders ⋈ order_items aggregated to order grain
  Aggregate FIRST, then left join: joining the line-level table directly would multiply the 3.42M order rows into 33.8M and break every order-level average. Left (not inner) keeps the 75,000 test orders, whose basket_size is legitimately null.


**Impact on record count: zero, on every join.** That is the result we wanted and the
reason the checks are shown. Every basket line found its product and its order; no line
was duplicated by a one-to-many fan-out; the 75,000 test orders survive the left join with
a null basket size rather than disappearing through an inner join.

### Cleaning log — the complete audit trail

In [13]:
dl = pd.DataFrame(log["decisions"])
ic.savetable(dl, "cleaning_decisions")
for i, d in enumerate(log["decisions"], 1):
    print(f"\n[{i}] {d['step']}\n    action: {d['action']}\n    why:    {d['reason']}\n    impact: {d['impact']}")

  table  -> reports/tables/cleaning_decisions.csv  (11 rows)

[1] categories
    action: Flagged department 'missing' (21) and aisles 'missing' (100) / 'other' (6) as placeholders instead of deleting them.
    why:    They are unknown-category buckets, not real shelves. Dropping them would silently delete real transactions; keeping them unflagged would let an 'unknown' bucket win a 'top category' ranking.
    impact: 1 department + 2 aisle rows flagged; 0 rows removed.

[2] products.product_name
    action: Trimmed/collapsed whitespace in product names (4 rows had stray whitespace).
    why:    Whitespace variants create false distinct products when names are grouped or matched.
    impact: 4 names normalised; 0 rows removed.

[3] products.product_uid
    action: Created product_uid: product_ids that share an identical name *within the same aisle* now map to one analytical product (111 groups, 112 secondary SKUs folded in). product_id is kept untouched.
    why:    e.g. 'BBQ Sauce' (26

## 2.8 The analysis-ready model

In [14]:
shapes = pd.DataFrame([{"table": k, "rows": v[0], "columns": v[1]} for k, v in log["final_shapes"].items()])
display(shapes)

,table,rows,columns
0,aisles_clean,134,3
1,departments_clean,21,3
2,products_clean,49688,10
3,orders_clean,3421083,14
4,customers,206209,15
5,order_items,33819106,13


In [15]:
oc = ic.load_clean("orders_clean")
cu = ic.load_clean("customers")
it = ic.load_clean("order_items", columns=["order_id","user_id","product_uid","aisle_id","department_id","reordered"])
print("order_items (the fact table) — one row per product per order:")
display(it.head(5))
print("\norders_clean — one row per order:")
display(oc.head(3))
print("\ncustomers — one row per customer, fully derived (there is no users table in the source):")
display(cu.head(3))

order_items (the fact table) — one row per product per order:


,order_id,user_id,product_uid,aisle_id,department_id,reordered
0,2,202279,33120,86,16,1
1,2,202279,28985,83,4,1
2,2,202279,9327,104,13,0
3,2,202279,45918,19,13,1
4,2,202279,30035,17,13,0



orders_clean — one row per order:


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,is_first_order,dspo_is_capped,order_dow_label,basket_size,n_reordered,has_basket_data,basket_reorder_rate
0,2539329,1,prior,1,2,8,NaN,True,False,Mon (2),5.0,0.0,True,0.0
1,2398795,1,prior,2,3,7,15.0,False,False,Tue (3),6.0,3.0,True,0.5
2,473747,1,prior,3,3,12,21.0,False,False,Tue (3),5.0,3.0,True,0.6



customers — one row per customer, fully derived (there is no users table in the source):


,user_id,n_orders_total,mean_dspo,median_dspo,fav_dow,fav_hour,n_orders_observed,total_items,avg_basket,median_basket,max_basket,items_reordered,reorder_rate,distinct_products,repeat_breadth
0,1,11,19.000000,19.5,4,7,11,70.0,6.363636,6.0,11.0,51.0,0.728571,19,0.271429
1,2,15,16.285715,13.0,1,10,15,226.0,15.066667,14.0,31.0,105.0,0.464602,121,0.535398
2,3,13,12.000000,11.0,0,16,12,88.0,7.333333,7.0,11.0,55.0,0.625000,33,0.375000


In [16]:
# Final integrity assertions: the data model has to hold before any analysis is run on it.
checks = [
 ("every basket line resolves to a product", bool((it.product_uid>0).all())),
 ("every basket line resolves to an aisle", bool((it.aisle_id>0).all())),
 ("every basket line resolves to a department", bool((it.department_id>0).all())),
 ("every basket line resolves to a customer", bool((it.user_id>0).all())),
 ("order_items row count unchanged by integration", len(it)==33_819_106),
 ("orders_clean row count unchanged", len(oc)==3_421_083),
 ("one row per customer", cu.user_id.is_unique),
 ("customer count matches orders", cu.user_id.nunique()==oc.user_id.nunique()),
 ("only test orders lack basket data", int((~oc.has_basket_data).sum())==75_000),
 ("items total reconciles with per-order basket sizes", int(oc.basket_size.sum())==len(it)),
]
display(pd.DataFrame(checks, columns=["assertion","passes"]))
assert all(c[1] for c in checks), "integration integrity failed"
print("\nAll integration assertions pass. Phases 2 and 3 complete — 0 rows deleted, 0 values imputed.")

,assertion,passes
0,every basket line resolves to a product,True
1,every basket line resolves to an aisle,True
2,every basket line resolves to a department,True
3,every basket line resolves to a customer,True
4,order_items row count unchanged by integration,True
5,orders_clean row count unchanged,True
6,one row per customer,True
7,customer count matches orders,True
8,only test orders lack basket data,True
9,items total reconciles with per-order basket sizes,True



All integration assertions pass. Phases 2 and 3 complete — 0 rows deleted, 0 values imputed.
